# الدرس 11 - بروتوكول وكيل إلى وكيل (A2A)


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ما هو بروتوكول A2A؟

بروتوكول **Agent-to-Agent (A2A)** هو معيار مفتوح يمكّن وكلاء الذكاء الاصطناعي من التواصل،
اكتشاف بعضهم البعض، والتعاون — حتى عندما يكونون مبنيين على أُطر عمل مختلفة أو مستضافين
من قِبل خدمات مختلفة.

Key concepts:

- **Discovery** – يقوم الوكلاء بنشر *Agent Card* التي تصف قدراتهم، مما يجعل من السهل على وكلاء آخرين (أو المنسقين) العثور على الاختصاصي المناسب لمهمة.
- **Message Passing** – يتبادل الوكلاء رسائل مُهيكلة عبر بروتوكول مشترك، بحيث يمكن فهم طلب من وكيل واحد وتنفيذه بواسطة آخر بغض النظر عن تنفيذه الداخلي.
- **Task Lifecycle** – يعرّف A2A حالات مثل *submitted*، *working*، *completed*، و*failed*، مما يمنح المنسق رؤية كاملة حول كيفية تقدم المهمة المفوَّضة.

في هذا الدرس نقوم بمحاكاة تعاون بنمط A2A عن طريق ربط ثلاثة وكلاء سفر متخصصين
في سير عمل حيث يساهم كل وكيل بخبرته ويمرر النتائج إلى التالي.


## إنشاء وكلاء سفر متخصصين


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## تعاون متعدد الوكلاء عبر سير العمل

نقوم بربط الوكلاء الثلاثة في سير عمل متتابع يعكس تمرير رسائل A2A:

1. **CurrencyExchangeAgent** يتلقى طلب المستخدم ويقدم إرشادات حول العملات.
2. **ActivityPlannerAgent** يتلقى السياق الموسع ويضيف توصيات بالأنشطة.
3. **TravelManagerAgent** يدمج كلا المدخلين في موجز سفر نهائي.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## فهم A2A في بيئة الإنتاج

في بيئة الإنتاج، يفتح بروتوكول A2A سيناريوهات قوية عبر الخدمات:

| القدرة | الوصف |
|---|---|
| **التشغيل البيني بين الأُطر** | يمكن لوكيل مبني بإطار عمل واحد أن يفوض مهامًا إلى وكيل مبني بأي إطار عمل آخر متوافق مع A2A، مما يمكّن التشغيل البيني الحقيقي بين المؤسسات. |
| **حدود الخدمة** | يمكن أن يعيش الوكلاء في خدمات مصغرة منفصلة، أو مناطق سحابية مختلفة، أو حتى ضمن مؤسسات مختلفة بينما يتعاونون بسلاسة. |
| **الاكتشاف الديناميكي** | يمكن لمنسق العمل أن يستعلم سجل بطاقة الوكيل أثناء وقت التشغيل للعثور على المتخصص الأنسب لمهمة فرعية معينة. |
| **البث والإشعارات الفورية** | يدعم A2A Server-Sent Events (SSE) لتحديثات التقدم في الوقت الحقيقي والإشعارات الفورية للمهام طويلة التشغيل. |

سير العمل الذي بنيناه أعلاه هو نسخة مبسطة تعمل داخل نفس العملية من هذا النمط. في نشرٍ حقيقي
سيكشف كل وكيل عن نقطة نهاية HTTP، وينشر بطاقة الوكيل، ويتواصل
عبر بروتوكول A2A JSON-RPC.


‏## الملخص

في هذا الدرس تعلمت:

1. **ما هو بروتوكول A2A** — معيار مفتوح للاكتشاف بين الوكلاء، والمراسلة،
   وإدارة المهام.
2. **كيفية إنشاء وكلاء متخصصين** — وكيل تحويل العملات، ووكيل مخطط الأنشطة،
   ومنسق مدير السفر.
3. **كيفية توصيل الوكلاء في سير عمل** — باستخدام `WorkflowBuilder` لنمذجة التتابع
   في تمرير الرسائل بين الوكلاء.
4. **كيف يعمل A2A في الإنتاج** — تمكين التعاون عبر الأطر، وعبر الخدمات
   مع الاكتشاف الديناميكي والتحديثات المتدفقة.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
إخلاء المسؤولية:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية Co‑op Translator (https://github.com/Azure/co-op-translator). بينما نسعى إلى الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المرجعي والموثوق. بالنسبة للمعلومات الحرجة، يُنصح بالاستعانة بترجمة بشرية احترافية. لن نكون مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
